# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [12]:
%pip install -q wikipedia wikipedia-api requests lxml pymorphy3 scikit-learn seaborn matplotlib pandas numpy
%pip install -q stanza spacy conllu wtpsplit mwparserfromhell fasttext datasets datatrove trafilatura

<class 'ValueError'>: Can't find a pure Python 3 wheel for 'wikipedia'.
See: https://pyodide.org/en/stable/usage/faq.html#why-can-t-micropip-find-a-pure-python-wheel-for-a-package
You can use `await micropip.install(..., keep_going=True)` to get a list of all packages with missing wheels.

In [ ]:
import os
import time
import json
import wikipedia
import numpy as np
import requests

wikipedia.set_user_agent('HSE Practice (liza33ka@gmail.com)')

MIN_INTERVAL = 0.35
_last_call = [0.0]


def _wiki_request(params):
    params['format'] = 'json'
    params.setdefault('action', 'query')
    for attempt in range(5):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        r = requests.get(
            wikipedia.wikipedia.API_URL,
            params=params,
            headers={'User-Agent': wikipedia.wikipedia.USER_AGENT},
            timeout=30,
        )
        if r.status_code == 429:
            retry_after = int(r.headers.get('Retry-After', 30))
            print(f'429 Too Many Requests, ждем {retry_after} с')
            time.sleep(retry_after)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


wikipedia.wikipedia._wiki_request = _wiki_request


def load_with_disambigution(page):
    try:
        p = wikipedia.page(page)
    except wikipedia.DisambiguationError as e:
        random_option = np.random.choice(e.options)
        p = wikipedia.page(random_option)
    return p


def get_texts_for_lang(lang, n=10):
    """Скачивает n случайных статей из википедии на языке lang."""
    wikipedia.set_lang(lang)
    wiki_content = []

    pages = wikipedia.random(n)

    for page_name in pages:
        try:
            page = load_with_disambigution(page_name)
        except Exception as e:
            print('Skipping page {}'.format(page_name), str(e).strip('\n'))
            continue

        wiki_content.append(f'{page.title}\n{page.content.replace("==", "")}')
        
    return wiki_content


LANGUAGES = ['de', 'el', 'ko', 'ar', 'ja']
LANG_NAMES = {
    'de': 'немецкий',
    'el': 'греческий',
    'ko': 'корейский',
    'ar': 'арабский',
    'ja': 'японский',
}
N_ARTICLES = 10
WIKI_CACHE = 'wiki_texts.json'


if os.path.exists(WIKI_CACHE):
    with open(WIKI_CACHE, encoding='utf-8') as f:
        wiki_texts = json.load(f)
    print(f'Загружен кэш: {WIKI_CACHE}')
else:
    wiki_texts = {}


for lang in LANGUAGES:
    if wiki_texts.get(lang):
        print(f'{lang} ({LANG_NAMES[lang]}): уже в кэше, '
              f'{len(wiki_texts[lang])} статей')
        continue
    try:
        texts = get_texts_for_lang(lang, n=N_ARTICLES)
    except Exception as e:
        print(f'ERROR ON {lang}: {str(e).strip()}')
        continue

    wiki_texts[lang] = texts
    print(f'{lang} ({LANG_NAMES[lang]}): скачано {len(texts)} статей, '
          f'{sum(len(t) for t in texts)} символов')


    with open(WIKI_CACHE, 'w', encoding='utf-8') as f:
        json.dump(wiki_texts, f, ensure_ascii=False)
    time.sleep(1)


print(f'\nИтого сохранено языков: {list(wiki_texts.keys())}')

In [13]:
import re
import random
import numpy as np
import pandas as pd
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def clean_text(text: str) -> str:
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    text = re.sub(r"\[\[.*?\]\]", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"'{2,}", "", text)
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text: str):
    # терминаторы: латиница/греческий/арабский + японские 。！？
    parts = re.split(r"(?<=[.!?。！？])\s*", text)
    return [p.strip() for p in parts if len(p.strip()) > 15]


rows = []
for lang, articles in wiki_texts.items():
    for art in articles:
        cleaned = clean_text(art)
        for sent in split_sentences(cleaned):
            rows.append({"text": sent, "lang": lang})

df = pd.DataFrame(rows)
print(f'Размерность фрейма {df.shape}')
print(f'Количество предложений для каждого языка: {df["lang"].value_counts()}')
#print(df.head()) #проверка норм ли предложения

<class 'NameError'>: name 'wiki_texts' is not defined

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE")

texts = df["text"].tolist()
true_labels = df["lang"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Размер эмбеддингов:", embeddings.shape)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

kmeans = KMeans(n_clusters=len(LANGUAGES), random_state=SEED, n_init=10)
pred_labels = kmeans.fit_predict(embeddings)

ari = adjusted_rand_score(true_labels, pred_labels)
print(f"ARI = {ari:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

cm = confusion_matrix(true_labels, pred_labels, labels=LANGUAGES)
cm_df = pd.DataFrame(cm, index=LANGUAGES,
                     columns=[f"c{i}" for i in range(len(LANGUAGES))])
print(cm_df)

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(LANGUAGES)))
for i, lang in enumerate(LANGUAGES):
    mask = np.array(true_labels) == lang
    plt.scatter(coords[mask, 0], coords[mask, 1],
                label=f"{lang} ({LANG_NAMES[lang]})",
                alpha=0.6, s=20, c=[colors[i]])
plt.legend()
plt.title(f"LaBSE embeddings, PCA. ARI = {ari:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
errors_idx = [i for i in range(len(texts)) if true_labels[i] != pred_labels[i]]
print(f"Всего ошибок: {len(errors_idx)} из {len(texts)}")

random.seed(SEED)
sample_idx = random.sample(errors_idx, min(3, len(errors_idx)))

for i in sample_idx:
    text = texts[i]
    true_l = true_labels[i]
    pred_c = pred_labels[i]

    dists = np.linalg.norm(embeddings[i] - kmeans.cluster_centers_, axis=1)
    dists_dict = {f"c{j}": round(float(d), 3) for j, d in enumerate(dists)}

    cluster_langs = Counter(
        true_labels[k] for k in range(len(texts)) if pred_labels[k] == pred_c
    )

    print("=" * 70)
    print(f"Текст: {text[:250]}...")
    print(f"Истинный язык: {true_l} ({LANG_NAMES[true_l]})")
    print(f"Предсказанный кластер: c{pred_c}")
    print(f"Расстояния до центроидов: {dists_dict}")
    print(f"Состав кластера c{pred_c}: {dict(cluster_langs)}")

## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga - 
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче). 
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги. 
Обязательно покажите примеры, которые находятся по вашим запросам. 


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры 
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [ ]:
from pathlib import Path
import urllib.request
from collections import Counter
from spacy.matcher import DependencyMatcher

import spacy
from spacy.tokens import Doc
from conllu import parse_incr


URL = (
    "https://raw.githubusercontent.com/"
    "UniversalDependencies/UD_Russian-Taiga/master/"
    "ru_taiga-ud-dev.conllu"
)

CONLLU_PATH = Path("ru_taiga-ud-dev.conllu")

if not CONLLU_PATH.exists():
    print("Загружаем корпус...")
    urllib.request.urlretrieve(URL, CONLLU_PATH)

print("Файл корпуса:", CONLLU_PATH.resolve())

nlp = spacy.blank("ru")

sentences = []

with CONLLU_PATH.open(encoding="utf-8") as f:
    for sent in parse_incr(f):
        rows = [
            token
            for token in sent
            if isinstance(token["id"], int)
        ]
        if not rows:
            continue
        rows = [
            token
            for token in rows
            if token["form"] is not None
            and token["lemma"] is not None
            and token["upos"] is not None
            and token["deprel"] is not None
            and token["head"] is not None
        ]

        if not rows:
            continue

        ids = [token["id"] for token in rows]

        id_to_i = {
            token_id: i
            for i, token_id in enumerate(ids)
        }

        words = [token["form"] for token in rows]
        lemmas = [token["lemma"] for token in rows]
        upos = [token["upos"] for token in rows]

        spaces = [True] * len(words)
        spaces[-1] = False

        heads = []
        deps = []

        for i, token in enumerate(rows):
            head_id = token["head"]
            if head_id == 0:
                heads.append(i)
            elif head_id in id_to_i:
                heads.append(id_to_i[head_id])
            else:
                heads = []
                break

            deps.append(token["deprel"].split(":")[0])

        if not heads:
            continue

        doc = Doc(
            nlp.vocab,
            words=words,
            spaces=spaces,
            heads=heads,
            deps=deps,
        )

        for token, lemma, pos in zip(doc, lemmas, upos):
            token.lemma_ = lemma
            token.pos_ = pos
        sentences.append(doc)

print("Загружено предложений:", len(sentences))



In [ ]:
# query 1 Ищем прилагательные, которые связаны с существительными отношением amod
# Результаты можно использовать для анализа словосочетаний, построения словарей и извлечения описательных характеристик.

amod_results = []

for sentence in sentences:
    for token in sentence:
        if token.dep_ != "amod":
            continue
        if token.pos_ != "ADJ":
            continue
        noun = token.head
        if noun.pos_ not in ("NOUN", "PROPN"):
            continue
        if noun.sent != token.sent:
            continue

        adjective_lemma = token.lemma_.lower()
        noun_lemma = noun.lemma_.lower()

        amod_results.append({
            "adjective": token.text,
            "noun": noun.text,
            "pair": f"{adjective_lemma} — {noun_lemma}",
            "sentence": sentence.text,
        })

print("==============QUERY 1==============")
print("\nВсего найденных сочетаний:", len(amod_results))

print("\nПервые 15 найденных сочетаний:")

for row in amod_results[:15]:
    print(
        "-",
        row["pair"],
        "|",
        row["sentence"]
    )


pair_counts = Counter(
    row["pair"]
    for row in amod_results
)

print("\nТоп-15 сочетаний по частоте:")

for pair, count in pair_counts.most_common(15):
    print(f"{pair}: {count}")

print("Количество извлечённых сочетаний:", len(amod_results))
print("Количество уникальных сочетаний:", len(pair_counts))

In [ ]:
#query 2 конструкции, в которых у одного глагола одновременно есть подлежащее (nsubj) и прямое дополнение (obj)
# может быть полезен для автоматического извлечения событий, построения графов отношений и поиска конструкций типа «кто что сделал»
svo_matcher = DependencyMatcher(nlp.vocab)

svo_pattern = [
    {
        "RIGHT_ID": "verb",
        "RIGHT_ATTRS": {"POS": "VERB"},
    },
    {
        "LEFT_ID": "verb",
        "REL_OP": ">",
        "RIGHT_ID": "subject",
        "RIGHT_ATTRS": {
            "DEP": "nsubj",
            "POS": {"IN": ["NOUN", "PROPN", "PRON"]},
        },
    },
    {
        "LEFT_ID": "verb",
        "REL_OP": ">",
        "RIGHT_ID": "object",
        "RIGHT_ATTRS": {
            "DEP": "obj",
            "POS": {"IN": ["NOUN", "PROPN", "PRON"]},
        },
    },
]

svo_matcher.add("SUBJECT_VERB_OBJECT", [svo_pattern])

svo_results = []

for doc in sentences:
    for match_id, token_ids in svo_matcher(doc):
        verb_i, subj_i, obj_i = token_ids

        verb = doc[verb_i]
        subj = doc[subj_i]
        obj = doc[obj_i]

        svo_results.append({
            "S": subj.text,
            "V": verb.text,
            "O": obj.text,
            "леммы": (
                f"{subj.lemma_} — "
                f"{verb.lemma_} — "
                f"{obj.lemma_}"
            ),
            "предложение": doc.text,
        })

print("==============QUERY 2==============")
print("Всего найденных конструкций:", len(svo_results))

for row in svo_results[:20]:
    print(
        f"- {row['S']} — {row['V']} — {row['O']} "
        f"| {row['предложение']}"
    )

print("\nТоп-15 лемматизированных троек:")

for triple, count in Counter(
    row["леммы"] for row in svo_results
).most_common(15):
    print(f"{triple}: {count}")    

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения? 

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя. 

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу? 
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет? 
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



Прочитав статью FineWeb2: One Pipeline to Scale Them All, я задумалась о том, что стоит за впечатляющими цифрами в 20 терабайт и 5 миллиардов документов. Почему одни датасеты делают модели производительнее, а другие — лишь увеличивают энтропию? Полагаю, что ответ кроется в одном из ключевых решений авторов — в отказе от универсального подхода в пользу адаптивного.

Наиболее убедительным для меня стал результат, представленный на Рисунке 2. Авторы демонстрируют, что качество данных не является бинарным («хорошо» или «плохо»), а зависит от контекста. Документы, не имеющие дубликатов (cluster size = 1), зачастую оказываются низкокачественными, как и те, что дублируются тысячи раз. Это наблюдение проблематизирует привычное представление о том, что «уникальность» — это всегда преимущество. Авторы предлагают использовать частоту дублирования как прокси-сигнал качества и проводят «регидратацию» — выборочное возвращение дубликатов. Это не просто технический приём, а концептуальный сдвиг: дубликаты — не всегда шум, иногда это ценный сигнал о значимости и полезности данных.

Однако, соглашаясь с этим, я не могу не задаться вопросом: что именно означает «качество» в контексте FineWeb2? Авторы используют фильтрацию на основе эвристик (Gopher Quality, FineWeb Quality), изначально разработанных для английского языка. Они адаптируют пороги с помощью методов вроде Quantile или MeanStd, но сама природа этих фильтров остаётся англоцентричной. Например, фильтр, удаляющий документы с малым количеством стоп-слов, может быть неэффективен для языков, где стоп-слова не столь явно выражены, или для агглютинативных языков, где одно слово может нести смысл целого предложения. В статье упоминается, что для некоторых языков (например, суахили) оптимальный порог LID-фильтрации составляет 0.3, что удаляет почти 65% данных. Это свидетельствует о том, что «качество» — понятие относительное и сильно зависит от структуры языка.

Мне представляется, что наиболее уязвимым местом в дизайне FineWeb2 является именно эта зависимость от английских эвристик. Если бы мне нужно было построить корпус для языка, не исследованного в статье (например, для какого-либо папуасского языка), я бы доверяла пайплайну лишь частично. Я бы ожидала проблем на этапе токенизации (в статье признаётся, что для многих языков используются «прокси-токенизаторы» из близких языковых семей, что может быть неточно) и на этапе фильтрации. Метод «10Tail», удаляющий 10% данных, может оказаться слишком агрессивным для языка с малым объёмом данных, где каждый документ на счету. Здесь я бы предложила гипотетический эксперимент: сравнить модели, обученные на данных, отфильтрованных с помощью адаптивных порогов, и на данных, где фильтрация проводилась вручную носителями языка. Если разница в производительности окажется статистически незначительной, это подтвердит эффективность автоматизации. Если же ручная фильтрация даст значительный прирост, это будет означать, что предложенный пайплайн всё ещё слишком груб для низкоресурсных языков.

Ещё один аспект, который привлёк моё внимание, — это проблема «покрытия» языков. Авторы заявляют о 1000+ языках, но, как честно признаётся в статье, 70% из них (1320 из 1868) имеют более половины документов из Библии или Википедии. Это не столько датасет, сколько «эхо-камера» из ограниченного набора источников. Здесь я бы провела границу иначе: если язык представлен только религиозными текстами, он не может считаться полноценно покрытым для предобучения, поскольку модель не увидит разнообразия синтаксиса и лексики. Возможно, стоило бы ввести метрику «разнообразия доменов» и указывать её рядом с объёмом данных. В противном случае мы рискуем создать модель, которая будет генерировать библейские стихи на языке, но не сможет поддержать бытовой разговор.

В заключение хочу отметить, что статья FineWeb2 — это важный шаг вперёд. Она показывает, что создание «хорошего» датасета — это не просто сбор гигабайтов, а тонкая настройка под каждый язык. Но, как и в любом деле, дьявол кроется в деталях. Самое ценное в этой работе — не сам датасет, а методология и честное признание ограничений. Я согласна с авторами в том, что адаптивный пайплайн — это будущее, но я также считаю, что для по-настоящему качественного многоязычного датасета необходима коллаборация с носителями языка, которые смогут уловить нюансы, недоступные автоматическим фильтрам – только тогда мы сможем говорить о действительно «хорошем» датасете, а не просто о «большом».